# HW3 — Two retrievers and a model: what RAG actually sends

**Name:Yergazyyeva Dilnaz**
**Student ID:s23068163**
**Group:9**
**Repository:**

## AI tool disclosure

I used Claude (Anthropic) to explain the assignment, to check the Part A arithmetic, to draft the code for Parts B and C, to fix setup problems in VS Code, and to help draft the written answers from my notebook outputs. I ran all the code myself. The numbers in the outputs are from my own runs, and I checked the text against them.

>

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

Tokenisation: lowercase, split at spaces and hyphens, drop punctuation, no stop-word removal.

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl =** (9 + 10 + 6) / 3 = 25 / 3 = **8.3333**

**Query terms:** how, does, rag, retrieve, information  (N = 3)

**TF, DF and IDF of each query term:**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | ln(3.5/0.5) = 1.9459 |
| does | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retrieve | 0 | 0 | 0 | 0 | 1.9459 |
| information | 0 | 0 | 0 | 0 | 1.9459 |

D2 contains `retrieval` and `retrieves`, but the exact token `retrieve` is in no document, so its TF is 0 everywhere.

**BM25 of each document:**

> Every term has f(t, d) = 0, so every term's contribution is idf · 0 · (k1 + 1) / (0 + …) = 0.
>
> - D1: 0 + 0 + 0 + 0 + 0 = **0**
> - D2: 0 + 0 + 0 + 0 + 0 = **0**
> - D3: 0 + 0 + 0 + 0 + 0 = **0**

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0 | 1 (tie) |
| D2 | 0 | 1 (tie) |
| D3 | 0 | 1 (tie) |

## A1 (b) — BM25 with the stems from README §3

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, **gener**, text, and, answer, questions | 9 |
| D2 | **retriev**, augmented, **gener**, **retriev**, relevant, documents, before, **gener**, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl =** 25 / 3 = **8.3333** (stemming does not change the lengths)

**Stemmed query terms:** how, doe, rag, retriev, inform

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| doe | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retriev | 0 | 2 | 0 | 1 | ln((3 − 1 + 0.5)/(1 + 0.5)) = ln(1.6667) = 0.5108 |
| inform | 0 | 0 | 0 | 0 | 1.9459 |

**BM25 of each document**, worked:

> Only `retriev` matches anything (f = 2 in D2).
>
> Length normalisation for D2: k1 · (1 − b + b · |d|/avgdl) = 1.5 · (0.25 + 0.75 · 10/8.3333) = 1.5 · (0.25 + 0.9) = 1.725
>
> D2: 0.5108 · 2 · (1.5 + 1) / (2 + 1.725) = 0.5108 · 5 / 3.725 = 2.5541 / 3.725 = **0.6857**
>
> D1: all f = 0, so **0**. D3: all f = 0, so **0**.

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0 | 2 (tie) |
| D2 | 0.6857 | 1 |
| D3 | 0 | 2 (tie) |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

> In (a) there is no first document: all three score 0 and tie. The query word `retrieve` never appears as an exact token, because D2 has `retrieval` and `retrieves`. `how`, `does`, `rag` and `information` appear nowhere either. So the zero scores are correct, not a mistake: BM25 matches exact tokens and knows nothing about word forms.
>
> In (b), after stemming, `retrieve`, `retrieves` and `retrieval` all become `retriev`, so D2 matches twice and takes first place with 0.6857. D1 and D3 stay at 0. Stemming fixed the mismatch, but even then D2 gets credit for only one of the five query terms. `RAG` is never written in any document.

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```

‖Query‖ = √(0.8² + 0.2² + 0.5²) = √(0.64 + 0.04 + 0.25) = √0.93 = 0.9644

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| Query · D1 | 0.8·0.7 + 0.2·0.1 + 0.5·0.4 = 0.56 + 0.02 + 0.20 = 0.78 | 0.9644 | √(0.49 + 0.01 + 0.16) = √0.66 = 0.8124 | 0.78 / (0.9644 · 0.8124) = **0.9956** |
| Query · D2 | 0.8·0.9 + 0.2·0.3 + 0.5·0.6 = 0.72 + 0.06 + 0.30 = 1.08 | 0.9644 | √(0.81 + 0.09 + 0.36) = √1.26 = 1.1225 | 1.08 / (0.9644 · 1.1225) = **0.9977** |
| Query · D3 | 0.8·0.1 + 0.2·0.8 + 0.5·0.2 = 0.08 + 0.16 + 0.10 = 0.34 | 0.9644 | √(0.01 + 0.64 + 0.04) = √0.69 = 0.8307 | 0.34 / (0.9644 · 0.8307) = **0.4244** |

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D1 | 0.9956 | 2 |
| D2 | 0.9977 | 1 |
| D3 | 0.4244 | 3 |

**Which document is most similar to the query?**

> D2 (cosine 0.9977). It is the one about Retrieval-Augmented Generation, which is what the query asks about.

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

> The top document matches the stemmed BM25 (b): D2 first. The rest differs. Cosine gives a full ordering D2 > D1 > D3, while BM25 (b) puts D1 and D3 in a tie at 0. BM25 (a) gives no ranking at all (all zeros), so it disagrees with cosine completely. Note also that cosine for D1 (0.9956) and D2 (0.9977) are almost equal: the vectors say D1 is nearly as relevant as D2.

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

> BM25 needs the same token. The query says `retrieve` and `RAG`, but D2 says `retrieval`/`retrieves` and spells out "Retrieval-Augmented Generation", so without stemming BM25 finds nothing. An embedding model maps these to nearby vectors by meaning: `RAG` ≈ "Retrieval-Augmented Generation", `retrieve` ≈ `retrieves`, `information` ≈ `documents`. The flip side is that embeddings also rank D1 (about language models, answering questions) very high because it is topically close, even though it has none of the query's words.

Optional: check your Part A arithmetic here. The working above must still be written out by hand.

In [1]:
import math, re

docs = {
    'D1': 'Large language models can generate text and answer questions.',
    'D2': 'Retrieval-Augmented Generation retrieves relevant documents before generating an answer.',
    'D3': 'Embeddings represent text as numerical vectors.',
}
query = 'How does RAG retrieve information?'
stems = {'retrieve': 'retriev', 'retrieves': 'retriev', 'retrieval': 'retriev',
         'generate': 'gener', 'generating': 'gener', 'generation': 'gener',
         'information': 'inform', 'does': 'doe'}

def tok(s, stem=False):
    t = [w for w in re.split(r'[\s\-]+', re.sub(r'[^\w\s\-]', '', s.lower())) if w]
    return [stems.get(w, w) for w in t] if stem else t

def bm25_by_hand(stem, k1=1.5, b=0.75):
    T = {k: tok(v, stem) for k, v in docs.items()}
    q = tok(query, stem)
    N = len(T); L = {k: len(v) for k, v in T.items()}; avgdl = sum(L.values()) / N
    scores = {k: 0.0 for k in T}
    for t in q:
        n = sum(t in v for v in T.values()); idf = math.log((N - n + 0.5) / (n + 0.5))
        for k in T:
            f = T[k].count(t)
            scores[k] += idf * f * (k1 + 1) / (f + k1 * (1 - b + b * L[k] / avgdl))
    return L, avgdl, scores

for stem in (False, True):
    print('stemming' if stem else 'no stemming', bm25_by_hand(stem))

Q = [0.8, 0.2, 0.5]
for name, d in {'D1': [0.7, 0.1, 0.4], 'D2': [0.9, 0.3, 0.6], 'D3': [0.1, 0.8, 0.2]}.items():
    dot = sum(a * b for a, b in zip(Q, d))
    print(name, round(dot, 2), round(math.sqrt(sum(a*a for a in Q)), 4),
          round(math.sqrt(sum(a*a for a in d)), 4),
          round(dot / (math.sqrt(sum(a*a for a in Q)) * math.sqrt(sum(a*a for a in d))), 4))


no stemming ({'D1': 9, 'D2': 10, 'D3': 6}, 8.333333333333334, {'D1': 0.0, 'D2': 0.0, 'D3': 0.0})
stemming ({'D1': 9, 'D2': 10, 'D3': 6}, 8.333333333333334, {'D1': 0.0, 'D2': 0.6856719782093835, 'D3': 0.0})
D1 0.78 0.9644 0.8124 0.9956
D2 1.08 0.9644 1.1225 0.9977
D3 0.34 0.9644 0.8307 0.4244


---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [2]:
import os, json, re
from pathlib import Path
import numpy as np
from dotenv import load_dotenv

load_dotenv()                      # reads OPENAI_API_KEY and HF_TOKEN from .env (never printed)

DATA = Path('data') if Path('data/corpus.jsonl').exists() else Path('.')
corpus = [json.loads(l) for l in open(DATA / 'corpus.jsonl', encoding='utf-8')]
queries = json.load(open(DATA / 'queries.json', encoding='utf-8'))['queries']
qa = json.load(open(DATA / 'qa.json', encoding='utf-8'))['questions']

ids = [c['id'] for c in corpus]
texts = [c['text'] for c in corpus]
print(len(corpus), 'chunks,', len(queries), 'fixed queries,', len(qa), 'QA questions')


74 chunks, 8 fixed queries, 6 QA questions


## B1 — BM25 with `rank_bm25`

**My tokenisation:** lowercase: yes. Regex `\w+` (unicode, so Kazakh and Russian letters are kept as words). No stemming. No stop-word removal.

>

In [3]:
from rank_bm25 import BM25Okapi

def tokenize(s):
    return re.findall(r'\w+', s.lower())

bm25 = BM25Okapi([tokenize(t) for t in texts], k1=1.5, b=0.75)

def first_line(i, n=70):
    return texts[i].split('\n')[0][:n]

def bm25_top(q, k=3):
    s = bm25.get_scores(tokenize(q))
    order = np.argsort(-s)[:k]
    return [(int(i), float(s[i])) for i in order]

for q in queries:
    print(q['id'], q['text'])
    for r, (i, sc) in enumerate(bm25_top(q['text']), 1):
        print(f'  {r}. {ids[i]}  {sc:7.3f}  {first_line(i)}')


Q-01 What is form GR-305 used for?
  1. GO-08   15.825  Form GR-305 is used to appeal a decision of the Grant Office. It is th
  2. GO-12    8.195  A student who disagrees with a decision may appeal within 14 calendar 
  3. GO-06    8.070  Form GR-104 is the application form for the need-based study grant. It
Q-02 My family is poor. Is there financial help for students like me?
  1. KZ-17    8.726  The lake currently covers about 16,400 km2 (6,300 sq mi). However, lik
  2. GO-20    8.311  The need-based grant is for full-time undergraduate students only. Par
  3. KZ-25    7.511  Beshbarmak is usually made from finely chopped boiled meat, mixed with
Q-03 Өтінішті қашанға дейін тапсыру керек?
  1. GO-14   10.698  2026 жылғы грантқа өтініштер 1 қыркүйектен бастап 15 қазанға дейін сту
  2. KZ-01    0.000  Kazakhstan, officially the Republic of Kazakhstan, is a landlocked cou
  3. KZ-03    0.000  Kazakhstan has been inhabited since the Paleolithic era. In antiquity,
Q-04 What is the minimum

## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('google/embeddinggemma-300m', token=os.getenv('HF_TOKEN'))   # float32 (default)
doc_emb = model.encode_document(texts, show_progress_bar=False)
print('document embeddings:', doc_emb.shape)

def emb_top(q, k=3):
    qv = model.encode_query(q)
    s = model.similarity(qv, doc_emb)[0].cpu().numpy()
    order = np.argsort(-s)[:k]
    return [(int(i), float(s[i])) for i in order]

for q in queries:
    print(q['id'], q['text'])
    for r, (i, sc) in enumerate(emb_top(q['text']), 1):
        print(f'  {r}. {ids[i]}  {sc:7.3f}  {first_line(i)}')


c:\Users\user\ai hw\hw3-dilnazergazyyeva\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 314/314 [00:00<00:00, 314.95it/s]


document embeddings: (74, 768)
Q-01 What is form GR-305 used for?
  1. GO-08    0.696  Form GR-305 is used to appeal a decision of the Grant Office. It is th
  2. GO-12    0.461  A student who disagrees with a decision may appeal within 14 calendar 
  3. GO-23    0.450  Если студент не согласен с решением, он может подать апелляцию по форм
Q-02 My family is poor. Is there financial help for students like me?
  1. GO-01    0.445  The need-based study grant 2026 is run by the Grant Office for full-ti
  2. GO-10    0.412  Proof of income must accompany form GR-210. Accepted: a salary certifi
  3. GO-20    0.409  The need-based grant is for full-time undergraduate students only. Par
Q-03 Өтінішті қашанға дейін тапсыру керек?
  1. GO-14    0.399  2026 жылғы грантқа өтініштер 1 қыркүйектен бастап 15 қазанға дейін сту
  2. GO-09    0.350  Applications for the 2026 grant open on 1 September 2026 and close on 
  3. GO-11    0.301  If a required document is missing, the Grant Office does not ref

## My own queries

Written, with their `relevant` chunk ids, **before** running them.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | room 214 Grant Office | GO-21 | **BM25**: `214` is a rare exact token that appears only in GO-21, which BM25 weights heavily; an embedding model may blur a bare number. |
| Q-10 | I want to take a year off, will my money stop? | GO-18 | **Embeddings**: the query shares no words with GO-18 ("academic leave", "payments stop"); it needs paraphrase understanding. |

**What happened (after running):** both retrievers found the right chunk for both queries, so neither expectation became a clear win. Q-09: both put GO-21 first; BM25 gave it 12.367 against 4.903 for the next chunk, a big gap, so the rare token `214` did what I expected. Q-10: embeddings put GO-18 first (0.441), but BM25 also found it, in second place (7.872, just behind GO-13 with 8.311), so BM25 was less sure.

In [5]:
my_queries = [
    {'id': 'Q-09', 'text': 'room 214 Grant Office', 'relevant': ['GO-21']},
    {'id': 'Q-10', 'text': 'I want to take a year off, will my money stop?', 'relevant': ['GO-18']},
]
all_queries = queries + my_queries
print(len(all_queries), 'queries in total')


10 queries in total


## Results for all ten or more queries: top 3 from each retriever

In [6]:
def hit(top, relevant):
    return any(ids[i] in relevant for i, _ in top)

rows = []
for q in all_queries:
    b, e = bm25_top(q['text']), emb_top(q['text'])
    hb, he = hit(b, q['relevant']), hit(e, q['relevant'])
    rows.append((q['id'], [ids[i] for i, _ in b], hb, [ids[i] for i, _ in e], he))
    print(f"{q['id']}  {q['text']}   relevant={q['relevant']}")
    for name, top in (('BM25     ', b), ('Embedding', e)):
        for r, (i, sc) in enumerate(top, 1):
            print(f'   {name} {r}. {ids[i]}  {sc:7.3f}  {first_line(i, 60)}')
    print(f'   hit@3: BM25={hb}  Embedding={he}\n')

print('| Query | BM25 top 3 | BM25 hit@3 | Embedding top 3 | Embedding hit@3 |')
print('|---|---|:---:|---|:---:|')
for qid, b, hb, e, he in rows:
    print(f"| {qid} | {', '.join(b)} | {'✓' if hb else '✗'} | {', '.join(e)} | {'✓' if he else '✗'} |")
print(f"| **hit@3 total** | | {sum(r[2] for r in rows)}/{len(rows)} | | {sum(r[4] for r in rows)}/{len(rows)} |")


Q-01  What is form GR-305 used for?   relevant=['GO-08', 'GO-12', 'GO-23']
   BM25      1. GO-08   15.825  Form GR-305 is used to appeal a decision of the Grant Office
   BM25      2. GO-12    8.195  A student who disagrees with a decision may appeal within 14
   BM25      3. GO-06    8.070  Form GR-104 is the application form for the need-based study
   Embedding 1. GO-08    0.696  Form GR-305 is used to appeal a decision of the Grant Office
   Embedding 2. GO-12    0.461  A student who disagrees with a decision may appeal within 14
   Embedding 3. GO-23    0.450  Если студент не согласен с решением, он может подать апелляц
   hit@3: BM25=True  Embedding=True

Q-02  My family is poor. Is there financial help for students like me?   relevant=['GO-01', 'GO-02', 'GO-03']
   BM25      1. KZ-17    8.726  The lake currently covers about 16,400 km2 (6,300 sq mi). Ho
   BM25      2. GO-20    8.311  The need-based grant is for full-time undergraduate students
   BM25      3. KZ-25    7.511  Be

## B3 — comparison

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | GO-08, GO-12, GO-06 | ✓ | GO-08, GO-12, GO-23 | ✓ |
| Q-02 | KZ-17, GO-20, KZ-25 | ✗ | GO-01, GO-10, GO-20 | ✓ |
| Q-03 | GO-14, KZ-01, KZ-03 | ✓ | GO-14, GO-09, GO-11 | ✓ |
| Q-04 | GO-05, GO-17, GO-19 | ✗ | GO-02, GO-05, GO-17 | ✓ |
| Q-05 | KZ-05, KZ-06, KZ-11 | ✓ | KZ-05, KZ-06, KZ-03 | ✓ |
| Q-06 | KZ-37, KZ-38, KZ-03 | ✓ | KZ-37, KZ-38, KZ-10 | ✓ |
| Q-07 | GO-21, GO-16, GO-01 | ✗ | GO-15, GO-18, GO-01 | ✓ |
| Q-08 | GO-17, GO-05, GO-04 | ✓ | GO-17, GO-18, GO-15 | ✓ |
| Q-09 | GO-21, GO-02, GO-22 | ✓ | GO-21, GO-06, GO-19 | ✓ |
| Q-10 | GO-13, GO-18, KZ-41 | ✓ | GO-18, GO-17, GO-22 | ✓ |
| **hit@3 total** | | 7/10 | | 10/10 |

**Both retrievers agree:** which query, and why was it easy?

> Q-06 ("Where did the Soviet Union test its nuclear weapons?"). Both returned KZ-37 and then KZ-38 as the first two results. It was easy because the words of the question ("Soviet Union", "nuclear", "test") are written in almost the same way in those two chunks, so BM25 could match the words (scores 19.090 and 17.168) and the embedding model could match the meaning (0.596 and 0.510). Q-05 is similar: both put KZ-05 first because the names in the question appear in it.

**BM25 did better:** which query, and what exact string did it match that the embedding model blurred?

> In my run BM25 did not win any query on hit@3: where BM25 found the right chunk, the embeddings did too, and the embeddings also found it in 3 more queries (7/10 against 10/10). The best example of BM25 working as it should is Q-01. It matched the exact string `GR-305` and put GO-08 first with 15.825, almost double the next chunk (8.195). The embedding model also ranked GO-08 first (0.696), so it did not blur this string. The same happened with Q-09 and the number `214` (12.367 against 4.903). So in this corpus the exact-string advantage of BM25 was real, but the embedding model was never too weak to use it.

**Embeddings did better:** which query, and what could BM25 not match?

> Q-02 ("My family is poor. Is there financial help for students like me?"). BM25 returned KZ-17 (about a lake), GO-20 and KZ-25 (about food), so it found none of the relevant chunks GO-01, GO-02 and GO-03; it seems to have matched only ordinary words. Those chunks talk about a "low-income household" and a "need-based grant", and BM25 cannot connect "poor" with "low-income" because they are different words. The embeddings put GO-01 first (0.445) because the meaning is close. Q-07 is another case: the relevant chunk GO-15 is written in Russian and the question is in English, so they share no words and BM25 missed it. The embeddings ranked GO-15 first (0.422), but only just ahead of GO-18 (0.417), so that win was not very strong. In Q-04 BM25 put the archived chunk GO-05 first (12.780) because it contains the words "minimum GPA for the grant", while the right chunk GO-02 says "cumulative GPA is at least 2.67" and has no word "minimum".

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [7]:
import tiktoken
from openai import OpenAI

client = OpenAI()                  # uses OPENAI_API_KEY from the environment
MODEL = 'gpt-5.6-luna'
enc = tiktoken.get_encoding('o200k_base')

SYSTEM = '''Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].'''

def retrieve(method, question, k=3):
    top = bm25_top(question, k) if method == 'bm25' else emb_top(question, k)
    return [(corpus[i], sc) for i, sc in top]

def build_messages(question, chunks):
    context = '\n\n'.join(f"[{n}] {c['text']}" for n, c in enumerate(chunks, 1))
    user = f'Context:\n{context}\n\nQuestion: {question}'
    return [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': user}], context

def ask(question, method=None, k=3):
    if method is None:                                   # LLM only: no context at all
        messages = [{'role': 'user', 'content': question}]
    else:
        messages, _ = build_messages(question, [c for c, _ in retrieve(method, question, k)])
    resp = client.chat.completions.create(model=MODEL, messages=messages)
    return resp.choices[0].message.content, resp.usage


## C1 — the complete prompt, printed before it is sent

In [8]:
q = qa[0]['question']                      # C-01
chunks = [c for c, _ in retrieve('emb', q, 3)]
messages, context = build_messages(q, chunks)

print('=' * 70)
for m in messages:
    print(f"--- {m['role'].upper()} ---")
    print(m['content'])
print('=' * 70)

n_sys, n_q, n_ctx = len(enc.encode(SYSTEM)), len(enc.encode(q)), len(enc.encode(context))
print('System instructions :', n_sys)
print('Question            :', n_q)
print('Retrieved context   :', n_ctx)
print('Total input (tiktoken):', n_sys + n_q + n_ctx)
print('whole prompt text, tiktoken:', sum(len(enc.encode(m['content'])) for m in messages))
print('chunks:', len(chunks), '| tokens per chunk:', [len(enc.encode(c['text'])) for c in chunks],
      '| context size:', n_ctx, 'tokens,', len(context), 'characters')

resp = client.chat.completions.create(model=MODEL, messages=messages)
print('\nANSWER:', resp.choices[0].message.content)
print('usage.prompt_tokens =', resp.usage.prompt_tokens)


--- SYSTEM ---
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].
--- USER ---
Context:
[1] The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2] The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 2: from 85,000 up to 150,000 tenge per person per month. Band 3: above 150,000 tenge per person per month. Households in band 3 are not eligible for the need-based grant, although their students may still apply for merit scholarships, whic

| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | 44 |
| Question | 18 |
| Retrieved context | 288 |
| **Total input** | 350 |
| **`usage.prompt_tokens` from the response** | 364 |

- Number of retrieved chunks: 3 (GO-04, GO-03, GO-15: the amounts, the income bands, the payment rules)
- Tokens per chunk: 90, 104, 85
- Total context size (tokens / characters): 288 tokens / 1145 characters

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

> My three counts add up to 350, but the API says 364, so 14 tokens are missing. 4 of them come from the words I added around the parts ("Context:", "Question:" and the blank lines): when I counted the whole text of both messages together I got 354. The other 10 tokens (364 − 354) are added by the API itself to wrap each message (role names and separators) and to start the reply. I cannot see those tokens in my own text, but I pay for them.

**What exactly does retrieval add to the model's input?**

> Only text. Retrieval takes the top 3 chunks and pastes them as numbered blocks ([1], [2], [3]) into the user message, in front of the question. In my run that was 288 tokens (1145 characters), about 82% of the whole 350-token input. The model gets no index, no files and no search tool.

## C2 — three systems, six questions

In [9]:
methods = [('LLM only', None), ('BM25 RAG', 'bm25'), ('Embedding RAG', 'emb')]
c2 = {}
for item in qa:
    for name, m in methods:
        ans, usage = ask(item['question'], m)
        c2[(item['id'], name)] = ans

for item in qa:
    print('=' * 70)
    print(item['id'], item['question'])
    print('REFERENCE:', item['reference'])
    for name, _ in methods:
        print(f'--- {name} ---')
        print(c2[(item['id'], name)])


C-01 How much does a student in income band 2 receive, and when is it paid?
REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.
--- LLM only ---
Which student funding scheme and country do you mean? “Income band 2” is used by several systems, and the amount and payment date vary by location and academic year.
--- BM25 RAG ---
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments: 75,000 tenge in October and 75,000 tenge in February**. [1]
--- Embedding RAG ---
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: one in **October** and one in **February**. [1]
C-02 My grant application was refused. How long do I have to appeal, and which form do I use?
REFERENCE: 14 calendar days from the date on the decision letter, using form GR-305.
--- LLM only ---
The appeal deadline and form depend on the **grant program, agen

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | ✗ asks "which scheme and country?" | ✓ 150,000 tenge, October and February | ✓ 150,000 tenge, October and February |
| C-02 | ✗ generic advice, no days, no form | ✓ 14 calendar days, form GR-305 | ✓ 14 calendar days, form GR-305 |
| C-03 | ✗ asks which grant | ✓ 2.67 on the 4.0 scale | ✓ 2.67 on the 4.0 scale |
| C-04 | ✓ October 2017, moved from 2025 to 2031 | ✓ October 2017, 2025 → 2031 | ✓ October 2017, 2025 → 2031 |
| C-05 | ✓ 1963, Alma-Ata Institute of National Economy | ✓ 1963, Almaty Institute of National Economy | ✓ 1963, Almaty Institute of National Economy |
| C-06 | ✗ long answer about US grants, no "I do not know" | ✓ "I do not know", separate scheme | ✓ "I don't know", separate scheme |
| **agrees with `reference`** | 2/6 | 6/6 | 6/6 |

The LLM alone was right only on the two questions about public Wikipedia facts (C-04, C-05), which it probably saw in training. On the four questions about the handbook (C-01, C-02, C-03, C-06) it could not know the answer.

## C3 — question 1 of 3: C-03

**1. What BM25 retrieved** (top 3, ids and scores):

> GO-16 (15.799), GO-02 (14.059), GO-06 (13.852).

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> GO-02 (0.629), GO-20 (0.569), GO-05 (0.545).

**3. What was sent to the model** (the context blocks):

> BM25 context: [1] GO-16 (the 400,000 tenge cap and holding a merit scholarship), [2] GO-02 (the three conditions, including "cumulative GPA is at least 2.67 on the 4.0 scale"), [3] GO-06 (form GR-104).
> Embedding context: [1] GO-02, [2] GO-20 (full-time undergraduates only), [3] GO-05 (the ARCHIVED 2025 rule with the old minimum GPA of 3.0).

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> It changed the answer from useless to correct. The model alone only asked "Which need-based grant do you mean?". With either retriever it said 2.67 on a 4.0 scale and cited the block that holds the rule ([2] for BM25, [1] for embeddings). The answers are more specific and grounded in the handbook.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> Nothing was missed in the final answer. There was a small retrieval problem: BM25 put the wrong chunk first (GO-16, 15.799, only a little above GO-02 with 14.059), and the embedding context contained the trap chunk GO-05 with the old minimum GPA of 3.0. The generation part did well: the model did not use the old number and picked the 2026 rule. The marking "ARCHIVED" in GO-05 and the system instruction to use the context both helped. If the right chunk had been missing from the context, the model could not have answered correctly. Note that for the query Q-04 (similar words) BM25 did not return GO-02 at all, and with the longer wording of C-03 it did, so BM25 depends on the exact words of the question.

In [10]:
qid = 'C-03'
item = next(x for x in qa if x['id'] == qid)
print(item['question'], '\nREFERENCE:', item['reference'], '\nholding chunks:', item['relevant'])
for name, m in (('BM25', 'bm25'), ('EmbeddingGemma', 'emb')):
    res = retrieve(m, item['question'], 3)
    print(f'\n=== {name} top 3 ===')
    for r, (c, sc) in enumerate(res, 1):
        print(f"{r}. {c['id']}  score={sc:.3f}")
    _, ctx = build_messages(item['question'], [c for c, _ in res])
    print(f'--- context sent to the model ({name}) ---')
    print(ctx)
    print('--- answer ---')
    print(c2[(qid, 'BM25 RAG' if m == 'bm25' else 'Embedding RAG')])


What is the minimum GPA needed to qualify for the need-based grant? 
REFERENCE: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.) 
holding chunks: ['GO-02']

=== BM25 top 3 ===
1. GO-16  score=15.799
2. GO-02  score=14.059
3. GO-06  score=13.852
--- context sent to the model (BM25) ---
[1] The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[2] An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Office. Meeting two

## C3 — question 2 of 3: C-06

**1. What BM25 retrieved** (top 3, ids and scores):

> GO-16 (16.996), GO-01 (14.173), GO-20 (13.720).

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> GO-20 (0.666), GO-16 (0.616), GO-01 (0.554).

**3. What was sent to the model** (the context blocks):

> BM25: [1] GO-16 (cap and merit scholarship), [2] GO-01 (the grant is for full-time undergraduates), [3] GO-20 (undergraduates only; "Master's and doctoral students are supported by a separate scheme, which this handbook does not describe").
> Embedding: [1] GO-20, [2] GO-16, [3] GO-01. The same three chunks in a different order.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> This question has no answer in the corpus, so the correct answer is "I do not know". The model alone wrote a long answer about US grants (Pell grants, FAFSA), which does not fit this university. With retrieval it said "I do not know how much you can get" (BM25, cited [3]) and "I don't know the amount" (embeddings, cited [1][3]), and explained that master's students have a separate scheme. So retrieval made the answer more grounded and stopped the guessing.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> Nothing was missed: the amount is not in the handbook, so no retriever could find it. GO-20 was retrieved by both (rank 3 for BM25, rank 1 for embeddings), and both answers used it. This is a limit of the data, not a retrieval or generation problem. The test showed that the instruction "If the answer cannot be found in the context, say that you do not know" works, because both systems followed it.

In [11]:
qid = 'C-06'
item = next(x for x in qa if x['id'] == qid)
print(item['question'], '\nREFERENCE:', item['reference'], '\nholding chunks:', item['relevant'])
for name, m in (('BM25', 'bm25'), ('EmbeddingGemma', 'emb')):
    res = retrieve(m, item['question'], 3)
    print(f'\n=== {name} top 3 ===')
    for r, (c, sc) in enumerate(res, 1):
        print(f"{r}. {c['id']}  score={sc:.3f}")
    _, ctx = build_messages(item['question'], [c for c, _ in res])
    print(f'--- context sent to the model ({name}) ---')
    print(ctx)
    print('--- answer ---')
    print(c2[(qid, 'BM25 RAG' if m == 'bm25' else 'Embedding RAG')])


I am a master's student. How much need-based grant can I get? 
REFERENCE: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe. 
holding chunks: ['GO-20']

=== BM25 top 3 ===
1. GO-16  score=16.996
2. GO-01  score=14.173
3. GO-20  score=13.720
--- context sent to the model (BM25) ---
[1] The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[2] The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a student from a low-income household can pay for study costs without taking on paid work that harms their studies. One grant is awarded per student per academic year. The grant is a gift, n

## C3 — question 3 of 3: C-02

**1. What BM25 retrieved** (top 3, ids and scores):

> GO-05 (17.241), GO-08 (14.529), GO-12 (13.573).

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> GO-08 (0.603), GO-12 (0.559), GO-11 (0.534).

**3. What was sent to the model** (the context blocks):

> BM25: [1] GO-05 (the ARCHIVED 2025 rules), [2] GO-08 ("Form GR-305 is used to appeal a decision"), [3] GO-12 ("may appeal within 14 calendar days of the date on the decision letter, using form GR-305").
> Embedding: [1] GO-08, [2] GO-12, [3] GO-11 (a missing document, 10 working days to upload it).

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> Without retrieval the model could only give general advice ("depends on which grant scheme and country") and ask for details. With both retrievers it gave the exact numbers: 14 calendar days from the date on the decision letter and form GR-305. Retrieval made the answer accurate and specific, even though each retriever also brought one wrong chunk.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> There was a retrieval problem that did not hurt the answer. BM25 put the archived chunk GO-05 first (17.241), higher than the two chunks with the answer, and it did not return GO-23 (the Russian appeal chunk). The embeddings returned GO-11 (0.534), which is about another deadline (10 working days) and could have confused the model. In both cases the generation was fine: the BM25 answer cited [3] (GO-12) and the embedding answer cited [1][2] (GO-08 and GO-12), so the model used the right blocks and ignored the wrong ones. This shows that a wrong chunk in the top 3 is not always a problem, but it is a risk: if the question had been about the minimum GPA, the archived chunk could have given a wrong answer.

In [12]:
qid = 'C-02'
item = next(x for x in qa if x['id'] == qid)
print(item['question'], '\nREFERENCE:', item['reference'], '\nholding chunks:', item['relevant'])
for name, m in (('BM25', 'bm25'), ('EmbeddingGemma', 'emb')):
    res = retrieve(m, item['question'], 3)
    print(f'\n=== {name} top 3 ===')
    for r, (c, sc) in enumerate(res, 1):
        print(f"{r}. {c['id']}  score={sc:.3f}")
    _, ctx = build_messages(item['question'], [c for c, _ in res])
    print(f'--- context sent to the model ({name}) ---')
    print(ctx)
    print('--- answer ---')
    print(c2[(qid, 'BM25 RAG' if m == 'bm25' else 'Embedding RAG')])


My grant application was refused. How long do I have to appeal, and which form do I use? 
REFERENCE: 14 calendar days from the date on the decision letter, using form GR-305. 
holding chunks: ['GO-08', 'GO-12', 'GO-23']

=== BM25 top 3 ===
1. GO-05  score=17.241
2. GO-08  score=14.529
3. GO-12  score=13.573
--- context sent to the model (BM25) ---
[1] ARCHIVED — kept for reference only. Under the 2025 scheme the minimum GPA for the grant was 3.0, band 1 received 200,000 tenge, band 2 received 120,000 tenge, and the whole amount was paid in a single instalment in November. These rules were replaced on 1 June 2026 and do not apply to any application made for the 2026 academic year.

[2] Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

> BM25 compares words: it counts how often the words of the query appear in a chunk and gives more weight to rare words, so the chunk must contain the same words. Embedding retrieval turns the query and every chunk into a vector of numbers (768 numbers in my run) that represents meaning, and finds the chunks whose vectors are closest. So BM25 matches the exact words and embeddings match the meaning.

**2. When can BM25 perform better than embeddings?**

> When the query has a rare exact string such as a code, a name or a number. In my run BM25 found form `GR-305` (Q-01, score 15.825 against 8.195 for the next chunk) and `214` (Q-09, 12.367 against 4.903) with a clear gap. BM25 is also fast and needs no model. In my run, however, the embeddings found these chunks too, so BM25 never won on hit@3.

**3. When can embeddings perform better than BM25?**

> When the question uses different words from the text: synonyms, a paraphrase or another language. In Q-02 "poor" has to be connected with "low-income", and BM25 returned a lake and food chunks while the embeddings returned GO-01 first. In Q-07 the right chunk is in Russian and the question is in English, and only the embeddings found it. Embeddings also found Q-04 (GO-02 first) where BM25 put the archived chunk first.

**4. Why is cosine similarity used with embeddings?**

> Cosine similarity measures the angle between two vectors, so it compares the direction of the vectors, which carries the meaning, and not their length. The score is easy to compare between chunks (in my run the embedding scores were between about 0.29 and 0.70). In Part A, D2 got cosine 0.9977 and D3 only 0.4244.

**5. What information is actually sent to an LLM in a RAG system?**

> A text message: the system instructions, the retrieved chunks pasted as numbered blocks, and the question. In my C1 example this was 44 tokens of instructions, 18 of question and 288 of context (350 in total, and the API counted 364).

**6. Does the LLM search the original documents itself?**

> No. The model only sees the text that my code puts in the prompt. The search is done by BM25 or the embedding model before the model is called. This is why the LLM alone could not answer the handbook questions (it got only 2/6 in C2, both on Wikipedia facts).

**7. What happens if retrieval returns an irrelevant chunk?**

> It wastes tokens and it can mislead the model, or the model ignores it. In C-02 BM25 put the archived chunk GO-05 first, but the model used GO-12 and answered correctly. If the right chunk is missing completely, the model cannot give the right answer from the context: it should say "I do not know" (as in C-06) or it may guess.

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

> More chunks give a bigger chance that the needed chunk is included, but they also add more tokens (more cost) and more irrelevant text that can distract the model. I tested C-01. K=1: both retrievers returned GO-04, 93 context tokens, 169 prompt tokens, correct answer. K=3: BM25 returned GO-04, GO-02, GO-01 (291 context tokens, 367 prompt tokens) and the embeddings GO-04, GO-03, GO-15 (288 and 364), correct answer. K=8: 721 and 797 tokens for BM25, 730 and 806 for embeddings; the answer was still correct, even though the context now included the archived chunk GO-05. So in this question a bigger K did not make the answer worse, but it cost about 8 times more context tokens (93 to 721-730) for the same answer. K=1 was enough here because the answer is in one chunk. For a question that needs several chunks (like the appeal question C-02, with three relevant chunks) K=1 would not be enough.

In [13]:
qq = next(x for x in qa if x['id'] == 'C-01')
for K in (1, 3, 8):
    for m in ('bm25', 'emb'):
        res = retrieve(m, qq['question'], K)
        msgs, ctx = build_messages(qq['question'], [c for c, _ in res])
        r = client.chat.completions.create(model=MODEL, messages=msgs)
        print(f"K={K} {m:4s} chunks={[c['id'] for c, _ in res]} context_tokens={len(enc.encode(ctx))} "
              f"prompt_tokens={r.usage.prompt_tokens}")
        print('   ', r.choices[0].message.content.replace('\n', ' ')[:300])
print('reference:', qq['reference'])


K=1 bm25 chunks=['GO-04'] context_tokens=93 prompt_tokens=169
    A student in income band 2 receives **150,000 tenge** for the academic year, paid in **two equal instalments of 75,000 tenge**: the first in **October** and the second in **February**. [1]
K=1 emb  chunks=['GO-04'] context_tokens=93 prompt_tokens=169
    A student in income band 2 receives **150,000 tenge per academic year**, paid in **two equal instalments of 75,000 tenge**: the first in **October** and the second in **February**. [1]
K=3 bm25 chunks=['GO-04', 'GO-02', 'GO-01'] context_tokens=291 prompt_tokens=367
    A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: **75,000 tenge in October** and **75,000 tenge in February**. [1]
K=3 emb  chunks=['GO-04', 'GO-03', 'GO-15'] context_tokens=288 prompt_tokens=364
    A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: the first in **Oct

**9. Is a vector database required to build a RAG system?**

> No. In this homework the document vectors were just a NumPy array of shape (74, 768) and I compared the query with all of them in a loop. BM25 does not need vectors at all. A vector database becomes useful when there are very many chunks and a full comparison is too slow.

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.

> Embeddings worked better for retrieval: hit@3 was 10/10 for EmbeddingGemma and 7/10 for BM25 (BM25 failed on Q-02, Q-04 and Q-07, which need meaning, a paraphrase or another language). In the final answers (C2) the two RAG systems were equal: 6/6 for BM25 RAG and 6/6 for Embedding RAG, against 2/6 for the LLM alone. The reason is that in those six questions BM25 still had the needed chunk in its top 3 (for example GO-02 at rank 2 in C-03), and the model ignored the wrong chunks. Six questions is a small sample, so the better retrieval of embeddings (10/10) is the stronger evidence.

---
# Conclusion

> I learned that RAG is a search step plus a model: the model sees only the text that retrieval puts into the prompt. In my results RAG was clearly better than the LLM alone (6/6 against 2/6) on the handbook questions, which the model had never seen. BM25 is good for exact strings like `GR-305` and `214`, but it failed when the question used other words or another language (7/10), while EmbeddingGemma found the right chunk for all 10 queries. Both systems had wrong chunks in the top 3 sometimes (the archived chunk GO-05), and the model usually ignored them, but this is a risk. Next time I would try to combine both retrievers and check the answers on more questions.